# Task 14: Primary Adjusted Analysis

This notebook estimates the adjusted association between documented sepsis and documented inpatient palliative-care use among adult hematologic-malignancy hospitalizations. Run all cells to refresh the model and tables.

## Model specification

Outcome: normalized `Z51.5` (`Z515`) in any diagnosis position. Exposure: documented sepsis (`A41*`). Covariates: continuous age, sex, race/ethnicity, primary payer, income quartile, cancer-excluded Charlson category, hospital region, hospital location/teaching, hospital bed size, admission year, and mutually exclusive HM subtype. Missing demographic categories are modeled explicitly.

The logistic model uses `DISCWT`. Variance uses year-specific `NIS_STRATUM` with discharge-level Taylor linearization. Since `HOSP_NIS` is unavailable by study decision, confidence intervals and p-values are strata-adjusted approximations rather than full NIS hospital-cluster-adjusted estimates. Adjusted probabilities are average marginal predictions over the observed weighted covariate distribution.

In [1]:
from pathlib import Path
import sys
import pandas as pd
from IPython.display import display
REPO_ROOT = Path.cwd()
if REPO_ROOT.name == 'notebooks': REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path: sys.path.insert(0, str(REPO_ROOT))
from src.phase_8_primary_adjusted import main
summary = main()

{
  "outcome": "Documented inpatient palliative-care use (normalized Z515 in any diagnosis position)",
  "included_unweighted_records": 994992,
  "excluded_records": 0,
  "iterations": 8,
  "reference_categories": {
    "sex": "Male",
    "race": "White",
    "payer": "Medicare",
    "income": "0\u201325th percentile",
    "cci_category": "0",
    "region": "Northeast",
    "location_teaching": "Rural",
    "bed_size": "Small",
    "year_category": "2016",
    "hm_subtype_label": "Lymphoma"
  },
  "variance_note": "DISCWT-weighted logistic model with year-specific NIS_STRATUM linearization and discharge-level variance units; not full NIS hospital-cluster-adjusted inference.",
  "primary_results": [
    {
      "measure": "Adjusted odds ratio for documented sepsis",
      "estimate": 2.801,
      "ci_95": "2.756\u20132.848",
      "p_value": "<0.001"
    },
    {
      "measure": "Adjusted absolute probability difference, percentage points",
      "estimate": 9.46,
      "ci_95": "9.27\

## Primary adjusted results

In [2]:
primary = pd.read_csv(REPO_ROOT / 'outputs/phase_8/primary_adjusted_results.csv', keep_default_na=False)
primary.columns = ['Measure', 'Estimate', '95% CI', 'P-value']
display(primary.style.hide(axis='index'))

Measure,Estimate,95% CI,P-value
Adjusted odds ratio for documented sepsis,2.801,2.756–2.848,<0.001
"Adjusted absolute probability difference, percentage points",9.46,9.27–9.65,<0.001
Total,—,—,—


## Adjusted palliative-care probabilities

In [3]:
probabilities = pd.read_csv(REPO_ROOT / 'outputs/phase_8/adjusted_probabilities.csv', keep_default_na=False)
probabilities.columns = ['Sepsis status', 'Adjusted probability, %', '95% CI lower, %', '95% CI upper, %']
display(probabilities.style.hide(axis='index'))

Sepsis status,"Adjusted probability, %","95% CI lower, %","95% CI upper, %"
No documented sepsis,6.58,6.53,6.63
Documented sepsis,16.04,15.86,16.22
Total observed cohort,—,—,—


## Reference categories and analytic cohort

In [4]:
references = pd.DataFrame(summary['reference_categories'].items(), columns=['Variable', 'Reference category'])
references.loc[len(references)] = ['Total analytic records', f"{summary['included_unweighted_records']:,}"]
display(references.style.hide(axis='index'))

Variable,Reference category
sex,Male
race,White
payer,Medicare
income,0–25th percentile
cci_category,0
region,Northeast
location_teaching,Rural
bed_size,Small
year_category,2016
hm_subtype_label,Lymphoma


## Full coefficient audit table

This supporting table is included for model review. The prespecified primary result is the sepsis estimate above.

In [5]:
coefficients = pd.read_csv(REPO_ROOT / 'outputs/phase_8/full_model_coefficients.csv', keep_default_na=False)
coefficients.columns = ['Term', 'Adjusted odds ratio', '95% CI lower', '95% CI upper', 'P-value']
display(coefficients.style.hide(axis='index'))

Term,Adjusted odds ratio,95% CI lower,95% CI upper,P-value
Intercept,0.022,0.021,0.023,<0.001
Documented sepsis,2.801,2.756,2.848,<0.001
"Age, per year (centered at 65)",1.035,1.034,1.036,<0.001
sex: Female,1.019,1.004,1.035,0.013
sex: Missing,0.947,0.569,1.576,0.835
race: Black,0.97,0.946,0.994,0.016
race: Hispanic,0.919,0.892,0.946,<0.001
race: Asian/Pacific Islander,0.877,0.837,0.919,<0.001
race: Native American,0.981,0.868,1.108,0.754
race: Other,0.856,0.816,0.897,<0.001


## Copy/paste-friendly Markdown

In [6]:
def print_markdown(dataframe, title):
    print(f'## {title}\n')
    headers = list(dataframe.columns)
    print('| ' + ' | '.join(headers) + ' |')
    print('|' + '|'.join(['---'] * len(headers)) + '|')
    for row in dataframe.astype(str).itertuples(index=False, name=None):
        print('| ' + ' | '.join(value.replace('|', '\\|') for value in row) + ' |')
    print()
print_markdown(primary, 'Primary adjusted results')
print_markdown(probabilities, 'Adjusted palliative-care probabilities')

## Primary adjusted results

| Measure | Estimate | 95% CI | P-value |
|---|---|---|---|
| Adjusted odds ratio for documented sepsis | 2.801 | 2.756–2.848 | <0.001 |
| Adjusted absolute probability difference, percentage points | 9.46 | 9.27–9.65 | <0.001 |
| Total | — | — | — |

## Adjusted palliative-care probabilities

| Sepsis status | Adjusted probability, % | 95% CI lower, % | 95% CI upper, % |
|---|---|---|---|
| No documented sepsis | 6.58 | 6.53 | 6.63 |
| Documented sepsis | 16.04 | 15.86 | 16.22 |
| Total observed cohort | — | — | — |



## Interpretation

After adjustment for the prespecified demographic, socioeconomic, comorbidity, hospital, year, and malignancy-subtype covariates, documented sepsis remained associated with higher documented inpatient palliative-care use. This observational association does not establish that sepsis caused palliative-care use.